In [1]:
import os
import requests
import uuid
import json
from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()


False

In [2]:
from groq import Groq

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables.")

client = Groq(api_key=api_key)


In [3]:
# Session data structures
sessions = {}
current_session = None
new_url = ""


In [4]:
def fetch_website_content(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        paragraphs = soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
        text_content = ' '.join([element.get_text().strip() for element in paragraphs])
        return text_content if text_content else None
    except Exception as e:
        print(f"Error fetching website content: {str(e)}")
        return None

def process_content(text, max_length=28000):
    cleaned_text = ' '.join(text.split())
    return cleaned_text[:max_length] if len(cleaned_text) > max_length else cleaned_text


In [5]:
def generate_chat_response(user_input, context, history=[]):
    try:
        messages = [
            {
                "role": "system",
                "content": f"Answer questions using only this context: {context}. do not use the out of webpage informations, just say out of context info. try to use minimum tokens with all nessesory info."
            }
        ]
        for exchange in history:
            messages.append({"role": "user", "content": exchange['user']})
            if exchange['bot']:
                messages.append({"role": "assistant", "content": exchange['bot']})
        messages.append({"role": "user", "content": user_input})
        chat_completion = client.chat.completions.create(
            messages=messages,
            model="llama3-70b-8192",
            temperature=0.4,
            max_tokens=150
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error generating response: {str(e)}"


In [6]:
def create_session(url, context):
    session_id = str(uuid.uuid4())
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {
        'id': session_id,
        'url': url,
        'context': context,
        'history': [],
        'created': timestamp,
        'last_accessed': timestamp
    }

def add_to_history(session, user_input, bot_response):
    session['history'].append({
        'user': user_input,
        'bot': bot_response,
        'timestamp': datetime.now().strftime("%H:%M:%S")
    })
    session['last_accessed'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return session




In [ ]:
from IPython.display import display, Markdown

# Prompt user for URL input
url = input("Enter Website URL (e.g., https://example.com): ")
if url.startswith("http"):
    raw_text = fetch_website_content(url)
    if raw_text:
        processed_text = process_content(raw_text)
        session = create_session(url, processed_text)
        sessions[session['id']] = session
        current_session = session['id']
        print(f"Session created for {url}")
    else:
        print("Could not fetch content.")
else:
    print("Invalid URL. Please start with http.")

# Start chat loop for current session
if current_session:
    session = sessions[current_session]
    while True:
        user_input = input("Your message (type 'exit' to stop): ")
        if user_input.lower() == 'exit':
            break
        bot_response = generate_chat_response(user_input, session['context'], session['history'])
        session = add_to_history(session, user_input, bot_response)
        sessions[current_session] = session
        display(Markdown(f"**You:** {user_input}\n\n**Assistant:** {bot_response}\n"))
